In [3]:
%pip install tensorflow
import collections
import random
import re
import tensorflow as tf
import requests

In [4]:
file = open('timemachine.txt', 'r')
data = file.read()
file.close()

In [5]:
def preprocess(text):
    return re.sub('[^A-Za-z]+', ' ', text).lower()

In [6]:
def tokenize(text):
    return text.split()

In [7]:
class Vocab:
    """Vocabulary for text."""
    def __init__(self, tokens=[], min_freq=0, reserved_tokens=[]):
        # Flatten a 2D list if needed
        if tokens and isinstance(tokens[0], list):
            tokens = [token for line in tokens for token in line]
        # Count token frequencies
        counter = collections.Counter(tokens)
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                  reverse=True)
        # The list of unique tokens
        self.idx_to_token = list(sorted(set(['<unk>'] + reserved_tokens + [
            token for token, freq in self.token_freqs if freq >= min_freq])))
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if hasattr(indices, '__len__') and len(indices) > 1:
            return [self.idx_to_token[int(index)] for index in indices]
        return self.idx_to_token[indices]

    @property
    def unk(self):  # Index for the unknown token
        return self.token_to_idx['<unk>']

In [8]:
def build(raw_text, vocab=None):
    """Return token indices and the vocabulary of the text."""
    tokens = tokenize(preprocess(raw_text))
    if vocab is None:
        vocab = Vocab(tokens)
    return [vocab[token] for token in tokens], vocab

In [9]:
corpus, vocab = build(data)
len(corpus), len(vocab)


(35810, 4928)

In [10]:
X, Y = [], []

for i in range(len(corpus) - 11):
    X.append(corpus[i:i + 10])
    Y.append(tf.one_hot(corpus[i + 10], len(vocab)))
X, Y = tf.convert_to_tensor(X), tf.convert_to_tensor(Y)

In [11]:
X.shape, Y.shape

(TensorShape([35799, 10]), TensorShape([35799, 4928]))

In [12]:
dataset = tf.data.Dataset.from_tensor_slices((X, Y))

In [13]:
class RNNModel(tf.keras.Model):
    def __init__(self, num_hiddens, vocab_size):
        super().__init__()
        self.rnn = tf.keras.layers.SimpleRNN(num_hiddens, return_sequences=False, return_state=True)
        self.vocab_size = vocab_size
        self.linear = tf.keras.layers.Dense(self.vocab_size)

    def one_hot(self, X):
        return tf.one_hot(X, self.vocab_size)

    def call(self, inputs):
        X = self.one_hot(inputs)
        outputs, H = self.rnn(X)
        return self.linear(outputs)
    
    def call_with_state(self, inputs, state):
        X = self.one_hot(inputs)
        outputs, H = self.rnn(X, initial_state=state)
        return self.linear(outputs), H

    def predict(self, prefix, num_preds, vocab, device=None):
        prefix = prefix.lower().split()
        state, outputs = None, [vocab[prefix[0]]]
        for i in range(len(prefix) + num_preds - 1):
            X = tf.constant([[outputs[-1]]])
            Y, state = self.call_with_state(X, state)
            if i < len(prefix) - 1:  # Warm-up period
                outputs.append(vocab[prefix[i + 1]])
            else:  # Predict num_preds steps
                outputs.append(int((tf.reshape(tf.argmax(Y, axis=1), 1)[0])))
        return ' '.join([vocab.idx_to_token[i] for i in outputs])


In [14]:
vocab_size = len(vocab)
num_hiddens = 32
num_steps = 10

model = RNNModel(num_hiddens, vocab_size)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.1, clipvalue=1.0), loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True))

In [15]:
model.predict('time traveller', 10, vocab)

'time traveller beast zenith mantel aloud intellect servant generations moderate lessons wind'

In [16]:
model.fit(X, Y, batch_size=1024, epochs=100, validation_batch_size=1024, validation_split=0.1)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 632ms/step - loss: 7.4109 - val_loss: 9.2411
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 623ms/step - loss: 6.8458 - val_loss: 9.6282
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 648ms/step - loss: 6.5172 - val_loss: 10.2741
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 31s 993ms/step - loss: 6.4744 - val_loss: 10.6563
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 619ms/step - loss: 6.2632 - val_loss: 10.6127
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 647ms/step - loss: 5.8821 - val_loss: 10.4823
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 657ms/step - loss: 5.7248 - val_loss: 10.2358
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 612ms/step - loss: 5.4519 - val_loss: 10.5989
Epoch 9/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 675ms/step - loss: 5.3530 - val_loss: 10.8173
Epoch 10/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 19s 580ms/step - loss: 5.1879 - val_loss: 11.1373
Epoch 11/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 619ms/step - loss: 5.0935 - val_loss: 11.2211
Epoch 12/100
32/32 ━━

In [20]:
model.predict('it is the best', 10, vocab)

'it is the best and i had come to push it to the time'